In [1]:
import sys
import torch
import kagglehub
import ultralytics

from ultralytics import YOLO

print("Python:", sys.version)
print("Python executable:", sys.executable)

print("Torch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)

print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = 0
else:
    DEVICE = "cpu"

print("Training device:", DEVICE)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/souravkumar/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Matplotlib is building the font cache; this may take a moment.


Python: 3.14.6 (v3.14.6:c63aec69bd5, Jun 10 2026, 08:07:54) [Clang 21.0.0 (clang-2100.1.1.101)]
Python executable: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14
Torch: 2.12.1
Ultralytics: 8.4.90
MPS available: True
MPS built: True
Training device: mps


In [5]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download(
    "mugheesahmad/sh17-dataset-for-ppe-detection"
)

DATASET_PATH = Path(path)

print("Path to dataset files:")
print(DATASET_PATH)

print("\nDataset exists:")
print(DATASET_PATH.exists())

print("\nDataset contents:")

for item in DATASET_PATH.iterdir():
    print(
        "[FOLDER]" if item.is_dir() else "[FILE]",
        item.name
    )

Resuming download from 13285457920 bytes (810833912 bytes left)...
Resuming download to /Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/1.archive (13285457920/14096291832) bytes left.


100%|█████████████████████████████████████████████████████████████████████████████| 13.1G/13.1G [02:10<00:00, 6.20MB/s]

Extracting files...


Path to dataset files:
/Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1

Dataset exists:
True

Dataset contents:
[FOLDER] voc_labels
[FOLDER] images
[FILE] train_files.txt
[FOLDER] labels
[FILE] val_files.txt
[FOLDER] meta-data


In [5]:
from pathlib import Path

DATASET_PATH = Path("/Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1")

IMAGES_PATH = DATASET_PATH / "images"
LABELS_PATH = DATASET_PATH / "labels"
TRAIN_FILE = DATASET_PATH / "train_files.txt"
VAL_FILE = DATASET_PATH / "val_files.txt"

print("Dataset:", DATASET_PATH.exists())
print("Images:", IMAGES_PATH.exists())
print("Labels:", LABELS_PATH.exists())
print("Train:", TRAIN_FILE.exists())
print("Val:", VAL_FILE.exists())

Dataset: True
Images: True
Labels: True
Train: True
Val: True


In [6]:
from pathlib import Path
import shutil

YOLO_DATASET = DATASET_PATH / "sh17_yolo11_dataset"

for split in ["train", "val"]:
    (YOLO_DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

def read_split(file_path):
    with open(file_path, "r") as f:
        return [line.strip() for line in f if line.strip()]

train_entries = read_split(TRAIN_FILE)
val_entries = read_split(VAL_FILE)

all_images = list(IMAGES_PATH.rglob("*"))
image_map = {img.name: img for img in all_images if img.is_file()}

def copy_split(entries, split):
    copied = 0
    missing = 0

    for entry in entries:
        img_name = Path(entry).name
        src_img = image_map.get(img_name)

        if src_img is None:
            missing += 1
            continue

        dst_img = YOLO_DATASET / "images" / split / img_name
        shutil.copy2(src_img, dst_img)

        src_label = LABELS_PATH / f"{src_img.stem}.txt"
        dst_label = YOLO_DATASET / "labels" / split / f"{src_img.stem}.txt"

        if src_label.exists():
            shutil.copy2(src_label, dst_label)
        else:
            dst_label.write_text("")

        copied += 1

    print(split, "copied:", copied, "missing:", missing)

copy_split(train_entries, "train")
copy_split(val_entries, "val")

train copied: 6479 missing: 0
val copied: 1620 missing: 0


In [7]:
import yaml

CLASS_NAMES = {
    0: "person",
    1: "ear",
    2: "ear-mufs",
    3: "face",
    4: "face-guard",
    5: "face-mask",
    6: "foot",
    7: "tool",
    8: "glasses",
    9: "gloves",
    10: "helmet",
    11: "hands",
    12: "head",
    13: "medical-suit",
    14: "shoes",
    15: "safety-suit",
    16: "safety-vest"
}

YOLO_YAML_PATH = YOLO_DATASET / "data.yaml"

yaml_config = {
    "path": YOLO_DATASET.resolve().as_posix(),
    "train": "images/train",
    "val": "images/val",
    "names": CLASS_NAMES
}

with open(YOLO_YAML_PATH, "w") as f:
    yaml.safe_dump(yaml_config, f, sort_keys=False)

print(YOLO_YAML_PATH)
print(YOLO_YAML_PATH.read_text())

/Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/sh17_yolo11_dataset/data.yaml
path: /Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/sh17_yolo11_dataset
train: images/train
val: images/val
names:
  0: person
  1: ear
  2: ear-mufs
  3: face
  4: face-guard
  5: face-mask
  6: foot
  7: tool
  8: glasses
  9: gloves
  10: helmet
  11: hands
  12: head
  13: medical-suit
  14: shoes
  15: safety-suit
  16: safety-vest



In [8]:
import torch

print("MPS available:", torch.backends.mps.is_available())

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

print("Using:", DEVICE)

MPS available: True
Using: mps


### try with one epoche

### Better training code

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path("/Users/souravkumar/Downloads/Human_Safety")

YOLO_YAML_PATH = Path(
    "/Users/souravkumar/.cache/kagglehub/datasets/"
    "mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/"
    "sh17_yolo11_dataset/data.yaml"
)

RUNS_DIR = PROJECT_ROOT / "runs" / "detect"

RUN_NAME = "sentineledge_yolo11m_full_batch8"

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

if torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)
print("Dataset YAML exists:", YOLO_YAML_PATH.exists())

if not YOLO_YAML_PATH.exists():
    raise FileNotFoundError(f"Missing YAML file: {YOLO_YAML_PATH}")

# ------------------------------------------------------------
# LOAD PRETRAINED MODEL
# ------------------------------------------------------------

model = YOLO("yolo11m.pt")

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

results = model.train(
    data=str(YOLO_YAML_PATH),

    # Training duration
    epochs=30,
    patience=8,

    # Input configuration
    imgsz=640,
    batch=8,

    # Hardware
    device=DEVICE,
    workers=0,
    amp=True,

    # Optimizer
    optimizer="AdamW",
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,

    # Augmentation
    mosaic=1.0,
    close_mosaic=10,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # Validation and outputs
    val=True,
    plots=True,
    save=True,
    save_period=1,

    # Reproducibility
    seed=42,
    deterministic=True,

    # Output path
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,

    verbose=True
)

Device: mps
Dataset YAML exists: True
New https://pypi.org/project/ultralytics/8.4.91 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.14.6 torch-2.12.1 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/sh17_yolo11_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.0

In [3]:
from pathlib import Path

LAST_MODEL_PATH = Path(
    "/Users/souravkumar/Downloads/Human_Safety/"
    "runs/detect/runs/"
    "sentineledge_yolo11m_full_batch8/"
    "weights/last.pt"
)

print("Checkpoint exists:", LAST_MODEL_PATH.exists())

if LAST_MODEL_PATH.exists():
    print("Checkpoint path:", LAST_MODEL_PATH)

Checkpoint exists: True
Checkpoint path: /Users/souravkumar/Downloads/Human_Safety/runs/detect/runs/sentineledge_yolo11m_full_batch8/weights/last.pt


In [1]:
from pathlib import Path
from ultralytics import YOLO

LAST_CHECKPOINT = Path(
    "/Users/souravkumar/Downloads/Human_Safety/"
    "runs/detect/"
    "sentineledge_yolo11m_full_batch8/"
    "weights/last.pt"
)

print("Checkpoint exists:", LAST_CHECKPOINT.exists())
print("Checkpoint path:", LAST_CHECKPOINT)

if not LAST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {LAST_CHECKPOINT}"
    )

# Load the interrupted training checkpoint
model = YOLO(str(LAST_CHECKPOINT))

# Resume the same training run
results = model.train(resume=True)

Checkpoint exists: True
Checkpoint path: /Users/souravkumar/Downloads/Human_Safety/runs/detect/sentineledge_yolo11m_full_batch8/weights/last.pt
New https://pypi.org/project/ultralytics/8.4.92 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.90 🚀 Python-3.14.6 torch-2.12.1 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/sh17_yolo11_dataset/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.

In [1]:
from pathlib import Path
from ultralytics import YOLO

BEST_MODEL_PATH = Path(
    "/Users/souravkumar/Downloads/Human_Safety/"
    "runs/detect/sentineledge_yolo11m_full_batch8/"
    "weights/best.pt"
)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best model not found: {BEST_MODEL_PATH}"
    )

best_model = YOLO(str(BEST_MODEL_PATH))

print("Best model loaded successfully")
print("Model path:", BEST_MODEL_PATH)

Best model loaded successfully
Model path: /Users/souravkumar/Downloads/Human_Safety/runs/detect/sentineledge_yolo11m_full_batch8/weights/best.pt


In [3]:
from pathlib import Path
from ultralytics import YOLO
import json

BEST_MODEL_PATH = Path(
    "/Users/souravkumar/Downloads/Human_Safety/"
    "runs/detect/sentineledge_yolo11m_full_batch8/"
    "weights/best.pt"
)

YOLO_YAML_PATH = Path(
    "/Users/souravkumar/.cache/kagglehub/datasets/"
    "mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/"
    "sh17_yolo11_dataset/data.yaml"
)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {BEST_MODEL_PATH}")

if not YOLO_YAML_PATH.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {YOLO_YAML_PATH}")

best_model = YOLO(str(BEST_MODEL_PATH))

metrics = best_model.val(
    data=str(YOLO_YAML_PATH),
    imgsz=640,
    batch=8,
    device="mps",
    workers=0,
    plots=True
)

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

f1_score = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0.0
)

metrics_summary = {
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1_score": round(f1_score, 4),
    "map50": round(map50, 4),
    "map50_95": round(map50_95, 4)
}

print(json.dumps(metrics_summary, indent=4))

Ultralytics 8.4.90 🚀 Python-3.14.6 torch-2.12.1 MPS (Apple M5)
YOLO11m summary (fused): 126 layers, 20,043,139 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2590.0±840.5 MB/s, size: 1619.5 KB)
val: Scanning /Users/souravkumar/.cache/kagglehub/datasets/mugheesahmad/sh17-dataset-for-ppe-detection/versions/1/sh17_yolo11_dataset/labels/val.cache... 1620 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1620/1620 226.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 203/203 1.3it/s 2:370.6ss
                   all       1620      15358      0.687      0.585      0.616      0.402
                person       1515       2734      0.868      0.895      0.914      0.752
                   ear        987       1612      0.857      0.782      0.794      0.498
              ear-mufs         38         49      0.554      0.286      0.366      0.254
                  face       1155      

In [8]:
from pathlib import Path

TEST_IMAGE = Path(
    "/Users/souravkumar/Downloads/Human_Safety/test_images/factory.jpg"
)

### Step 1: Run prediction

In [10]:
prediction_results = best_model.predict(
    source=str(TEST_IMAGE),
    conf=0.25,
    imgsz=640,
    device="mps",
    save=True
)


image 1/1 /Users/souravkumar/Downloads/Human_Safety/test_images/factory.jpg: 640x576 1 person, 2 ears, 1 face, 1 glasses, 1 helmet, 1 head, 2 safety-vests, 326.6ms
Speed: 13.2ms preprocess, 326.6ms inference, 43.5ms postprocess per image at shape (1, 3, 640, 576)
Results saved to /Users/souravkumar/Downloads/Human_Safety/runs/detect/predict


In [11]:
detections = []

for result in prediction_results:

    for box in result.boxes:

        class_id = int(box.cls[0])

        confidence = float(box.conf[0])

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        detections.append({

            "class_id": class_id,

            "class_name": result.names[class_id],

            "confidence": round(confidence, 3),

            "bounding_box": {
                "x1": round(x1, 1),
                "y1": round(y1, 1),
                "x2": round(x2, 1),
                "y2": round(y2, 1)
            }

        })

print(detections)

[{'class_id': 10, 'class_name': 'helmet', 'confidence': 0.949, 'bounding_box': {'x1': 1231.7, 'y1': 649.8, 'x2': 2517.4, 'y2': 1478.2}}, {'class_id': 0, 'class_name': 'person', 'confidence': 0.921, 'bounding_box': {'x1': 1304.8, 'y1': 582.6, 'x2': 3233.5, 'y2': 4083.8}}, {'class_id': 3, 'class_name': 'face', 'confidence': 0.782, 'bounding_box': {'x1': 1417.2, 'y1': 1181.7, 'x2': 2036.2, 'y2': 1882.9}}, {'class_id': 16, 'class_name': 'safety-vest', 'confidence': 0.697, 'bounding_box': {'x1': 1313.9, 'y1': 1661.7, 'x2': 2870.7, 'y2': 3925.4}}, {'class_id': 8, 'class_name': 'glasses', 'confidence': 0.695, 'bounding_box': {'x1': 1458.8, 'y1': 1186.5, 'x2': 1909.3, 'y2': 1465.3}}, {'class_id': 12, 'class_name': 'head', 'confidence': 0.665, 'bounding_box': {'x1': 1413.1, 'y1': 639.7, 'x2': 2446.8, 'y2': 1887.1}}, {'class_id': 1, 'class_name': 'ear', 'confidence': 0.324, 'bounding_box': {'x1': 2012.9, 'y1': 1265.5, 'x2': 2212.6, 'y2': 1593.4}}, {'class_id': 16, 'class_name': 'safety-vest', 'c

In [12]:
print(len(detections))

9
